# AGF-DDI — Auditable Graph-Fingerprint cold-start DDI
Runs on your frozen splits + the per-source drug feature tables. Dev splits are used only for early stopping/threshold; test splits are never used for training or tuning.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q rdkit lightgbm

In [ ]:
import os, subprocess
BASE = '/content/drive/MyDrive/Projects/auditddi'
CODE = '/content/auditddi'   # your git checkout
os.environ['AUDITDDI_DATA_BASE'] = BASE
# folder holding transductive_train.csv, validation.csv, transductive_test.csv, s1_dev.csv, s2_dev.csv, s1_test.csv, s2_test.csv
SPLITS = BASE + '/processed/<YOUR_FROZEN_SPLIT_FOLDER>'
OUT = BASE + '/results/agf_ddi'
for f in ['transductive_train','validation','transductive_test','s1_dev','s2_dev','s1_test','s2_test']:
    assert os.path.isfile(f'{SPLITS}/{f}.csv'), f'missing {f}.csv'
print('splits OK')

## 1. Which feature tables exist? (each dataset = one table, one row per drug)

In [ ]:
import glob
for p in sorted(glob.glob(BASE+'/processed/seven_branch/*.csv')+glob.glob(BASE+'/processed/auditddi_v2/source_feature_matrix.npz')):
    print(p)

## 2. Build any missing table (run only what is missing)

In [ ]:
SB = BASE + '/processed/seven_branch'
# !python {CODE}/scripts/build_pharmacogenomic_drug_profile.py --data-root {BASE}
# !python {CODE}/scripts/build_openfda_adverse_event_features.py --data-root {BASE} --output-dir {SB}
# !python {CODE}/scripts/build_chembl_protein_target_features.py --data-root {BASE} --chemreps {BASE}/chembl/chembl_37_chemreps.txt.gz --output-dir {SB}
# !python {CODE}/scripts/build_pdb_pocket_features.py --verified-matches <report.csv> --output-dir {SB}

## 3. Smoke test (about 1 minute)

In [ ]:
!python {CODE}/src/training/agf_ddi.py --splits_dir "{SPLITS}" --out_dir "{OUT}_smoke" --fast --seeds 1

## 4. Full run

In [ ]:
!python {CODE}/src/training/agf_ddi.py --splits_dir "{SPLITS}" --out_dir "{OUT}" --seeds 3 --ablate

In [ ]:
import json, pandas as pd
m = json.load(open(OUT+'/metrics.json'))
print(pd.DataFrame(m['metrics']).T[['regime','auroc','auprc','accuracy','f1','mcc','n']])
print(m['evidence_group_importance'])
print(pd.read_csv(OUT+'/ablation.csv'))